In [1]:
import pandas as pd
import requests
from datetime import datetime
import time

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []

start_year = datetime.now().year - 5
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f'{year}-{month:02d}-01'

        if month in [4, 6, 9, 11]:
            end_date = f'{year}-{month:02d}-30'
        elif month == 2:
            is_leap = (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0))
            end_date = f'{year}-{month:02d}-29' if is_leap else f'{year}-{month:02d}-28'
        else:
            end_date = f'{year}-{month:02d}-31'

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date
        }

        response = requests.get(url, params)

        if response.status_code != 200:
            print(f"Failed to retrieve data for {start_date} to {end_date}")
            continue

        try:
            data = response.json()
            for r in data.get('features', []):
                properties = r.get('properties', {})
                geometry = r.get('geometry', {})
                coordinates = geometry.get('coordinates', [None, None, None])

                lon = coordinates[0] if len(coordinates) > 0 else None
                lat = coordinates[1] if len(coordinates) > 1 else None
                depth = coordinates[2] if len(coordinates) > 2 else None

                record = {
                    'id': r.get('id'),
                    'time': properties.get('time'),
                    'updated': properties.get('updated'),
                    'latitude': lat,
                    'longitude': lon,
                    'depth_km': depth,
                    'mag': properties.get('mag'),
                    'magType': properties.get('magType'),
                    'place': properties.get('place'),
                    'status': properties.get('status'),
                    'tsunami': properties.get('tsunami'),
                    'alert': properties.get('alert'),
                    'felt': properties.get('felt'),
                    'cdi': properties.get('cdi'),
                    'mmi': properties.get('mmi'),
                    'sig': properties.get('sig'),
                    'net': properties.get('net'),
                    'code': properties.get('code'),
                    'ids': properties.get('ids'),
                    'sources': properties.get('sources'),
                    'types': properties.get('types'),
                    'nst': properties.get('nst'),
                    'dmin': properties.get('dmin'),
                    'rms': properties.get('rms'),
                    'gap': properties.get('gap'),
                    'type': properties.get('type'),
                }
                all_records.append(record)

            print(f"Retrieved {len(data.get('features', []))} records for {start_date} to {end_date}")
            time.sleep(0.5)

        except Exception as e:
            print(f"Failed to parse data for {start_date}: {e}")

df = pd.DataFrame(all_records)
display(df.head())

Retrieved 14812 records for 2021-01-01 to 2021-01-31
Retrieved 11449 records for 2021-02-01 to 2021-02-28
Retrieved 13126 records for 2021-03-01 to 2021-03-31
Retrieved 12444 records for 2021-04-01 to 2021-04-30
Retrieved 11186 records for 2021-05-01 to 2021-05-31
Retrieved 13792 records for 2021-06-01 to 2021-06-30
Retrieved 17454 records for 2021-07-01 to 2021-07-31
Retrieved 16343 records for 2021-08-01 to 2021-08-31
Retrieved 13257 records for 2021-09-01 to 2021-09-30
Retrieved 12054 records for 2021-10-01 to 2021-10-31
Retrieved 11183 records for 2021-11-01 to 2021-11-30
Retrieved 11043 records for 2021-12-01 to 2021-12-31
Retrieved 12491 records for 2022-01-01 to 2022-01-31
Retrieved 11284 records for 2022-02-01 to 2022-02-28
Retrieved 12346 records for 2022-03-01 to 2022-03-31
Retrieved 12019 records for 2022-04-01 to 2022-04-30
Retrieved 11801 records for 2022-05-01 to 2022-05-31
Retrieved 11445 records for 2022-06-01 to 2022-06-30
Retrieved 11826 records for 2022-07-01 to 2022

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [ ]:
import os

# Create the data folder if it doesn't exist under Google Drive
os.makedirs('/content/drive/MyDrive/global-seismic-trends/data', exist_ok=True)

# Save the DataFrame to a CSV inside the data folder
csv_path = '/content/drive/MyDrive/global-seismic-trends/data/earthquakes.csv'
df.to_csv(csv_path, index=False)

print(f"Successfully saved {len(df)} records to '{csv_path}'!")

Successfully saved 849573 records to '/content/drive/MyDrive/global-seismic-trends/data/earthquakes.csv'!
